<a href="https://colab.research.google.com/github/AhmedE16/flyrank-ai-intern-ahmed-essam/blob/main/work/notebooks/w01_research_question.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-02 — Research Question and Provisional Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w01_research_question.ipynb)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. My lane (or freestyle) and why

*Name your lane — or say 'freestyle' and describe your own question. One short paragraph: why this one?*

In [1]:
print("""
Lane: Refresh / Content Opportunity Scoring

I'm choosing this lane because Notebooks 01 and 02 already showed me its core
mechanic - a hand-written rule vs. a learned model, scored with Precision@K -
and I want to go deeper on that same problem rather than switch to something
new in Week 1. It also has the clearest link between a data signal, a decision
a reviewer makes, and an action someone takes, which is what this track is
asking me to reason about. I may adjust the exact label/window definition by
Week 4, since lane choice isn't locked yet.
""")


Lane: Refresh / Content Opportunity Scoring

I'm choosing this lane because Notebooks 01 and 02 already showed me its core
mechanic - a hand-written rule vs. a learned model, scored with Precision@K -
and I want to go deeper on that same problem rather than switch to something
new in Week 1. It also has the clearest link between a data signal, a decision
a reviewer makes, and an action someone takes, which is what this track is
asking me to reason about. I may adjust the exact label/window definition by
Week 4, since lane choice isn't locked yet.



## 2. The question: decision, action, cost of a wrong call

*What decision does your work improve? Who acts on it? What does a wrong recommendation cost?*

In [2]:
print("""
Unit of analysis: one page (one row = one content item, matching the starter
dataset's grain - content_id).

Decision this improves: today, a reviewer either works through pages in an
arbitrary order or relies on a hand-written rule like "stale + still getting
impressions." Both are weak - Notebook 01 showed the hand-written baseline
was only right ~24% of the time in its top 50 picks.

Action taken: a reviewer opens the top-ranked pages from my output and
decides whether to refresh, expand, protect, or deprioritize each one.

Cost of a wrong call:
- False positive (flagged, but wasn't really declining): wastes limited
  reviewer time.
- False negative (a truly declining page never surfaces): it keeps losing
  visibility unnoticed, and the client loses organic traffic in the meantime.
- Given limited reviewer capacity, missing a real decline is probably more
  expensive than reviewing a page that turns out fine.

Why data/ML can help: signals like impressions, CTR, position, and freshness
are cheap to compute across thousands of pages, while human judgment doesn't
scale past a handful. A ranked list lets a small review team spend limited
time on the pages most likely to matter - this is decision-support, not
automation.
""")


Unit of analysis: one page (one row = one content item, matching the starter
dataset's grain - content_id).

Decision this improves: today, a reviewer either works through pages in an
arbitrary order or relies on a hand-written rule like "stale + still getting
impressions." Both are weak - Notebook 01 showed the hand-written baseline
was only right ~24% of the time in its top 50 picks.

Action taken: a reviewer opens the top-ranked pages from my output and
decides whether to refresh, expand, protect, or deprioritize each one.

Cost of a wrong call:
- False positive (flagged, but wasn't really declining): wastes limited
  reviewer time.
- False negative (a truly declining page never surfaces): it keeps losing
  visibility unnoticed, and the client loses organic traffic in the meantime.
- Given limited reviewer capacity, missing a real decline is probably more
  expensive than reviewing a page that turns out fine.

Why data/ML can help: signals like impressions, CTR, position, and fresh

## 3. Quick look at the data (2-3 real numbers)

*Load the starter CSV below and show 2-3 real numbers that make your lane look worth the next 7 weeks.*

In [3]:
import os, sys, subprocess, json
import pandas as pd

IN_COLAB = "google.colab" in sys.modules
REPO_URL = "https://github.com/flyrank-bih/flyrank-ml-internship-starter"
REPO_DIR = "flyrank-ml-internship-starter"

if IN_COLAB:
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)

# Regenerate outputs/model_results.json if it doesn't exist yet
if not os.path.exists("outputs/model_results.json"):
    subprocess.run([sys.executable, "scripts/run_all.py"], check=True)

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
res = json.load(open("outputs/model_results.json"))

# Number 1: how much of the inventory is actually declining
declining_rate = (df["trend_direction"].str.lower() == "down").mean()
print(f"1) Declining rate across {len(df):,} pages: {declining_rate:.1%}")

# Number 2: the gap between hand-written rule and best model
base_p50 = res["baseline"]["baseline_precision_at_50"]
rf_p50 = res["models"]["random_forest"]["precision_at_50"]
print(f"2) Precision@50 — hand-written rule: {base_p50:.3f} vs random forest: {rf_p50:.3f}")
print(f"   -> the model gets roughly {rf_p50/base_p50:.1f}x more of its top-50 picks right")

# Number 3: how many pages are both declining AND still visible enough to matter
share_declining_and_visible = (
    (df["trend_direction"].str.lower() == "down") & (df["impressions_90d"] >= 500)
).mean()
print(f"3) Share of ALL pages both declining AND getting real traffic (impressions_90d >= 500): {share_declining_and_visible:.1%}")
print(f"   -> out of {len(df):,} pages, that's ~{round(share_declining_and_visible*len(df)):,} pages worth a reviewer's attention right now")

1) Declining rate across 30,000 pages: 54.2%
2) Precision@50 — hand-written rule: 0.240 vs random forest: 0.740
   -> the model gets roughly 3.1x more of its top-50 picks right
3) Share of ALL pages both declining AND getting real traffic (impressions_90d >= 500): 33.2%
   -> out of 30,000 pages, that's ~9,961 pages worth a reviewer's attention right now


## 4. Careful words: what I can and can't claim

*Write what your work will be able to say (observed, directional, decision-support) — and what it never will (causal proof, 'predicting Google').*

In [4]:
print("""
What I can claim: which pages, based on observed signals, look most similar
to pages that have historically declined - and rank them so a reviewer
spends limited time well.

What I cannot claim:
- That refreshing a flagged page will cause it to recover - that needs a
  real experiment, not this data.
- That any signal here reflects an actual Google ranking factor.
- That the starter label (trend_direction == "down") is a perfect target -
  it's a same-window proxy, not a true future outcome.
- Anything about AI search citations or rankings.

I'll use words like "observed," "associated with," and "this suggests" -
not "proves" or "causes."
""")


What I can claim: which pages, based on observed signals, look most similar
to pages that have historically declined - and rank them so a reviewer
spends limited time well.

What I cannot claim:
- That refreshing a flagged page will cause it to recover - that needs a
  real experiment, not this data.
- That any signal here reflects an actual Google ranking factor.
- That the starter label (trend_direction == "down") is a perfect target -
  it's a same-window proxy, not a true future outcome.
- Anything about AI search citations or rankings.

I'll use words like "observed," "associated with," and "this suggests" -
not "proves" or "causes."



## Self-check

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.